# PITT staircase — first look at the data

Opens one `{folder}_pitt.h5` written by spec-echem's PITT segment and makes the basic
review plots: the whole staircase, each step's current transient, the spectrum at the
end of every step, absorbance vs potential at one wavelength, and the spectra inside
one step.

Run each cell in order with **Shift-Enter**. Only cell 2 needs editing.

Needs only `numpy`, `pandas`, `matplotlib` and `h5py` — not spec-echem itself, so it
runs in any environment that can open an HDF5 file. The file layout is described in
spec-echem's `docs/data-format.md` §5 (and §4 for what every step group holds).

**This is a review, not the PITT analysis.** The charge column below is a plain
integral of the sampled current, which misses whatever flowed before each step's first
sample (`first_sample_s`) — treat it as a first look.

In [ ]:
import json
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

plt.rcParams.update({'font.size': 14})
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Settings — edit this cell

`data_path` can be the run folder (the one holding `{folder}_pitt.h5`) or the `.h5`
file itself. The `r` in front keeps Windows backslashes as they are.

- Windows: `r'C:\Users\inst-chem\Documents\specechem_data\20261007_pitt_test1'`
- Mac: `'/Users/name/data/20261007_pitt_test1'`

In [ ]:
data_path = r'COPY_PATH_TO_FOLDER_OR_H5_HERE'

wavelength = 800      # nm, for absorbance vs potential and the kinetics plots
step_to_plot = 2      # the step whose spectra-vs-time are drawn in section 6

save_figures = True   # PNGs go into the run folder, next to the .h5

## 2. Load the file

Every step is one group in the file. This reads them all, in step order, and prints one
row per step: its setpoint, how it ended, how long it was held, and how much data it
holds.

**`end_reason` is the first thing to read.** `cutoff` = the current settled to the set
fraction of its peak. **`max_hold` = it did NOT settle in the time allowed** — that step
is not at equilibrium. `fixed` = the optional end dedope. `stopped` / `aborted` = the
user interrupted there.

In [ ]:
path = Path(data_path)
if path.is_dir():
    found = sorted(path.glob('*_pitt.h5'))
    if not found:
        raise FileNotFoundError(f'No *_pitt.h5 in {path}')
    path = found[0]
folder = path.parent
print('Reading', path)

steps = []
with h5py.File(path, 'r') as f:
    wl = f['wavelength'][:]
    root = {k: f.attrs[k] for k in f.attrs}
    table = {s['index']: s for s in json.loads(root.get('pitt_steps_json', '[]'))}
    for key in sorted((k for k in f if k.isdigit()), key=int):
        g = f[key]
        a = dict(g.attrs)
        e = g['echem'] if 'echem' in g else None
        k = int(key)
        steps.append({
            'step': k,
            'role': str(a.get('role', '')),
            'E_set (V)': float(a.get('potential_set', np.nan)),
            'E_measured (V)': float(a.get('potential_measured', np.nan)),
            'end_reason': str(a.get('end_reason', '')),
            'hold (s)': float(a.get('hold_s', np.nan)),
            'peak |I| (A)': float(a.get('peak_current_A', np.nan)),
            'first_sample_s': table.get(k, {}).get('first_sample_s', np.nan),
            'n_spectra': g['absorbance_vs_time'].shape[1],
            'absorbance': g['absorbance_vs_time'][:].astype(float),   # (wavelength, time)
            't_spec': g['time'][:],
            't_echem': e['time'][:] if e is not None else np.array([]),
            't_pot': e['time_potentiostat'][:] if e is not None and 'time_potentiostat' in e else np.array([]),
            'I': e['current'][:] if e is not None else np.array([]),
            'E': e['potential'][:] if e is not None else np.array([]),
        })

# Rough charge per step: the trapezoid integral of the SAMPLED current (see the top).
for s in steps:
    ok = np.isfinite(s['I']) & np.isfinite(s['t_echem'])
    t, i = s['t_echem'][ok], s['I'][ok]
    # Written out rather than np.trapz, which newer numpy renames and older lacks the new name.
    s['Q rough (C)'] = float(np.sum(np.diff(t) * (i[1:] + i[:-1]) / 2)) if t.size > 1 else np.nan

summary = pd.DataFrame([{k: v for k, v in s.items() if not isinstance(v, np.ndarray)} for s in steps]).set_index('step')
print(f"{len(steps)} steps; every planned step ran: {bool(root.get('pitt_completed', False))}; "
      f"built by spec-echem {root.get('build_id', '?')}")
not_settled = summary.index[summary['end_reason'] == 'max_hold'].tolist()
if not_settled:
    print(f'NOT settled (max_hold): steps {not_settled}')
summary

In [ ]:
def save(fig, name):
    if save_figures:
        out = folder / f'{path.stem}_{name}.png'
        fig.savefig(out, dpi=150, bbox_inches='tight')
        print('saved', out.name)


def colors(n, cmap='viridis'):
    return plt.get_cmap(cmap)(np.linspace(0.05, 0.95, max(n, 1)))


def at_wavelength(target):
    i = int(np.argmin(np.abs(wl - target)))
    return i, wl[i]


def last_spectrum(s):
    # The end of the hold: the spectrum closest to equilibrium in that step.
    return s['absorbance'][:, -1] if s['absorbance'].shape[1] else np.full(wl.size, np.nan)


step_colors = colors(len(steps))

## 3. The whole staircase

Current (top) and measured potential (bottom) against the potentiostat's own clock,
which runs on across all steps — the cell was never switched off between them. One
colour per step.

In [ ]:
fig, (ax_i, ax_e) = plt.subplots(2, 1, sharex=True, figsize=(12, 8))
for s, c in zip(steps, step_colors):
    t = s['t_pot'] if s['t_pot'].size else s['t_echem']
    ax_i.plot(t, s['I'] * 1e6, color=c, lw=1.5)
    ax_e.plot(t, s['E'], color=c, lw=1.5)
ax_i.set_ylabel('Current (µA)')
ax_e.set_ylabel('Potential (V)')
ax_e.set_xlabel('Time since cell on (s)')
ax_i.set_title(path.stem)
fig.tight_layout()
save(fig, 'staircase')

## 4. Each step's current transient

|I| on a log scale against time within the step. A step that settled falls to its
cutoff; a capacitive-only step drops fast; a step still sloping at the end of its hold
(dashed = `max_hold`) had not reached equilibrium.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for s, c in zip(steps, step_colors):
    style = '--' if s['end_reason'] == 'max_hold' else '-'
    ax.semilogy(s['t_echem'], np.abs(s['I']), style, color=c, lw=1.5,
                label=f"{s['step']}: {s['E_set (V)']:+.3f} V ({s['role']})")
ax.set_xlabel('Time in step (s)')
ax.set_ylabel('|Current| (A)')
ax.set_title('Current transient per step   (dashed = did not settle)')
if len(steps) <= 20:
    ax.legend(fontsize=9, ncol=2)
save(fig, 'transients')

## 5. Spectra at the end of every step, and absorbance vs potential

Left: the last spectrum of each step — the one nearest equilibrium — coloured by
setpoint (dashed = `max_hold`). Right: absorbance at `wavelength` from those spectra
against the setpoint, forward and return legs marked separately, so hysteresis shows
directly.

In [ ]:
iw, wl_used = at_wavelength(wavelength)
fig, (ax_s, ax_v) = plt.subplots(1, 2, figsize=(15, 6))
for s, c in zip(steps, step_colors):
    style = '--' if s['end_reason'] == 'max_hold' else '-'
    ax_s.plot(wl, last_spectrum(s), style, color=c, lw=1.5)
ax_s.axvline(wl_used, color='gray', lw=0.8)
ax_s.set_xlabel('Wavelength (nm)')
ax_s.set_ylabel('Absorbance')
ax_s.set_title('Last spectrum of each step')

markers = {'forward': ('o', 'forward'), 'return': ('s', 'return'), 'dedope': ('^', 'end dedope')}
for role, (m, label) in markers.items():
    mine = [s for s in steps if s['role'] == role]
    if mine:
        ax_v.plot([s['E_set (V)'] for s in mine], [last_spectrum(s)[iw] for s in mine],
                  marker=m, ls='-' if role != 'dedope' else 'none', label=label)
ax_v.set_xlabel('Setpoint (V)')
ax_v.set_ylabel(f'Absorbance at {wl_used:.1f} nm')
ax_v.set_title('End-of-step absorbance vs potential')
ax_v.legend()
fig.tight_layout()
save(fig, f'spectra_and_A_vs_E_{wl_used:.0f}nm')

## 6. Spectra over time inside one step

Every spectrum of `step_to_plot`, light (start of the step) to dark (end). Spectra come
at full rate for the first seconds of a step, then more slowly.

In [ ]:
s = next(x for x in steps if x['step'] == step_to_plot)
fig, ax = plt.subplots(figsize=(10, 6))
cs = plt.get_cmap('bone_r')(np.linspace(0.25, 0.95, max(s['absorbance'].shape[1], 1)))
for j, c in enumerate(cs):
    ax.plot(wl, s['absorbance'][:, j], color=c, lw=1)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorbance')
ax.set_title(f"Step {s['step']} at {s['E_set (V)']:+.3f} V: {s['n_spectra']} spectra over "
             f"{s['t_spec'][-1] if s['t_spec'].size else 0:.1f} s ({s['end_reason']})")
save(fig, f"step{s['step']}_spectra_vs_time")

## 7. Kinetics at one wavelength, every step

Absorbance at `wavelength` against time within each step — the optical counterpart of
section 4. An optical trace still moving when the current has settled is worth noting:
the two need not settle together.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for s, c in zip(steps, step_colors):
    style = '--' if s['end_reason'] == 'max_hold' else '-'
    ax.plot(s['t_spec'], s['absorbance'][iw, :], style, color=c, lw=1.5,
            label=f"{s['E_set (V)']:+.3f} V")
ax.set_xlabel('Time in step (s)')
ax.set_ylabel(f'Absorbance at {wl_used:.1f} nm')
ax.set_title('Optical kinetics per step   (dashed = did not settle)')
if len(steps) <= 20:
    ax.legend(fontsize=9, ncol=2)
save(fig, f'kinetics_{wl_used:.0f}nm')

## 8. Change from the first step

The last spectrum of each step minus the last spectrum of step 0 — growth (polaron)
above zero, bleach (π–π*) below. Makes small changes between neighbouring steps easier
to see than the raw spectra in section 5.

In [ ]:
base = last_spectrum(steps[0])
fig, ax = plt.subplots(figsize=(10, 6))
for s, c in zip(steps[1:], step_colors[1:]):
    style = '--' if s['end_reason'] == 'max_hold' else '-'
    ax.plot(wl, last_spectrum(s) - base, style, color=c, lw=1.5)
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('ΔAbsorbance vs step 0')
ax.set_title(f"End-of-step spectra minus step 0 ({steps[0]['E_set (V)']:+.3f} V)")
save(fig, 'difference_spectra')